# BikeStore - Gold Layer Aggregation
BikeStore - Gold layer: data quality checks + business tables  
Outputs: daily_sales, monthly_sales, top_products, customer_lifetime_value  
Logs rows read / rejected / written at each step to gold/_pipeline_log  


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime

BASE   = "abfss://bikestore@olympicdata18.dfs.core.windows.net"
SILVER = f"{BASE}/silver"
GOLD   = f"{BASE}/gold"

log_rows = []

def log_step(step, rows_read, rows_written):
    rejected = rows_read - rows_written
    log_rows.append((datetime.now().isoformat(), step, rows_read, rejected, rows_written))
    print(f"{step:<40} read={rows_read:<8} rejected={rejected:<8} written={rows_written}")

In [ ]:
orders      = spark.read.format("delta").load(f"{SILVER}/sales/orders/")
order_items = spark.read.format("delta").load(f"{SILVER}/sales/order_items/")
customers   = spark.read.format("delta").load(f"{SILVER}/sales/customers/")
products    = spark.read.format("delta").load(f"{SILVER}/production/products/")

# Data type standardization
orders = (orders
    .withColumn("order_id", F.col("order_id").cast("int"))
    .withColumn("customer_id", F.col("customer_id").cast("int"))
    .withColumn("order_status", F.col("order_status").cast("int"))
    .withColumn("order_date", F.to_date("order_date"))
    .withColumn("required_date", F.to_date("required_date")))

order_items = (order_items
    .withColumn("order_id", F.col("order_id").cast("int"))
    .withColumn("product_id", F.col("product_id").cast("int"))
    .withColumn("quantity", F.col("quantity").cast("int"))
    .withColumn("list_price", F.col("list_price").cast("double"))
    .withColumn("discount", F.col("discount").cast("double")))

# Check 1: valid order dates
n = orders.count()
orders_ok = orders.filter(
    F.col("order_date").isNotNull() &
    (F.col("order_date") <= F.current_date()) &
    (F.col("required_date") >= F.col("order_date")))
log_step("orders: valid order dates", n, orders_ok.count())

# Check 2: no negative quantities/prices, discount between 0 and 1
n = order_items.count()
items_ok = order_items.filter(
    (F.col("quantity") > 0) &
    (F.col("list_price") > 0) &
    (F.col("discount").between(0, 1)))
log_step("order_items: valid qty/price/discount", n, items_ok.count())

# Check 3: referential integrity
n = items_ok.count()
items_ok = (items_ok
    .join(orders_ok.select("order_id"), "order_id", "inner")
    .join(products.select("product_id"), "product_id", "inner"))
log_step("order_items: valid FK to orders/products", n, items_ok.count())

# Check 4: exclude rejected orders (status 3)
n = orders_ok.count()
orders_ok = orders_ok.filter(F.col("order_status") != 3)
log_step("orders: exclude rejected status", n, orders_ok.count())

# Base fact table
sales_fact = (items_ok
    .join(orders_ok, "order_id")
    .withColumn("revenue", F.round(
        F.col("quantity") * F.col("list_price") * (1 - F.col("discount")), 2)))
sales_fact.cache()
log_step("sales_fact (joined)", items_ok.count(), sales_fact.count())

In [ ]:
# Gold tables
daily_sales = (sales_fact
    .groupBy("order_date")
    .agg(F.countDistinct("order_id").alias("total_orders"),
         F.sum("quantity").alias("total_units"),
         F.round(F.sum("revenue"), 2).alias("total_revenue"))
    .orderBy("order_date"))

monthly_sales = (sales_fact
    .withColumn("year", F.year("order_date"))
    .withColumn("month", F.month("order_date"))
    .groupBy("year", "month")
    .agg(F.countDistinct("order_id").alias("total_orders"),
         F.sum("quantity").alias("total_units"),
         F.round(F.sum("revenue"), 2).alias("total_revenue"))
    .orderBy("year", "month"))

top_products = (sales_fact
    .groupBy("product_id")
    .agg(F.sum("quantity").alias("units_sold"),
         F.round(F.sum("revenue"), 2).alias("total_revenue"))
    .join(products.select("product_id", "product_name", "brand_id", "category_id"), "product_id")
    .withColumn("revenue_rank", F.dense_rank().over(Window.orderBy(F.desc("total_revenue"))))
    .orderBy("revenue_rank"))

clv = (sales_fact
    .groupBy("customer_id")
    .agg(F.countDistinct("order_id").alias("total_orders"),
         F.round(F.sum("revenue"), 2).alias("lifetime_value"),
         F.min("order_date").alias("first_order_date"),
         F.max("order_date").alias("last_order_date"))
    .join(customers.select("customer_id", "first_name", "last_name", "city", "state"),
          "customer_id", "left")
    .orderBy(F.desc("lifetime_value")))

In [ ]:
# Write Gold + save the run log
gold_tables = {
    "daily_sales": daily_sales,
    "monthly_sales": monthly_sales,
    "top_products": top_products,
    "customer_lifetime_value": clv,
}

fact_rows = sales_fact.count()
for name, df in gold_tables.items():
    (df.write.mode("overwrite")
       .option("overwriteSchema", "true")
       .format("delta").save(f"{GOLD}/{name}/"))
    log_step(f"gold.{name}", fact_rows, df.count())

log_df = spark.createDataFrame(
    log_rows, ["run_time", "step", "rows_read", "rows_rejected", "rows_written"])
log_df.write.mode("append").format("delta").save(f"{GOLD}/_pipeline_log/")
display(log_df)